# EDFB - Digital Finance & Banking - Linear Probability Model and Logistic Regression


---


The following script provides examples on how to model binary outcomes using both linear probability models and logistic regression in R. We'll start with the linear probability model to understand its limitations, then move to logistic regression as a more appropriate approach for binary dependent variables. In order to run the script, you need to download the dataset "banking.csv".




In [ ]:
# Function to install packages if they are not already installed
install_if_missing <- function(packages) {
  for (pkg in packages) {
    if (!require(pkg, character.only = TRUE, quietly = TRUE)) {
      cat("Installing package:", pkg, "\n")
      install.packages(pkg, dependencies = TRUE, repos = "https://cran.rstudio.com/")
      library(pkg, character.only = TRUE)
    }
  }
}

# List of required packages
required_packages <- c(
  "dplyr",        # Data manipulation
  "ggplot2",      # Data visualization
  "readr",        # Reading CSV files
  "caret",        # Machine learning framework
  "corrplot",     # Correlation plots
  "gridExtra",    # Arranging plots
  "tidyr",        # Data tidying
  "scales",       # Scale functions for ggplot2
  "pROC",         # ROC curves
  "reshape2",     # Data reshaping
  "RColorBrewer", # Color palettes
  "viridis",      # Color scales
  "ROCR"          # ROC analysis
)

# Install missing packages and load all libraries
install_if_missing(required_packages)

cat("All required packages are installed and loaded successfully!\n")

In [ ]:
# To make this notebook's output stable across runs (we make the output reproducible)
set.seed(42)

## 0. Preparatory Steps

Before we dive into modeling binary outcomes, we need to prepare our data and understand its structure. This section covers:
- Loading and exploring the banking dataset
- Understanding the target variable (binary outcome)
- Preprocessing steps including handling categorical variables and scaling
- Addressing class imbalance through undersampling

### 0.1 Data Loading

We'll use a banking dataset that contains information about marketing campaigns for term deposits. Our goal is to predict whether a client will subscribe to a term deposit (binary outcome: yes/no).

In [ ]:
# Import real data from GitHub
banking_url <- "https://raw.githubusercontent.com/umatter/EDFB/main/data/banking.csv"
print("Fetching banking.csv from GitHub...")

In [ ]:
dataset <- read_csv(banking_url)

### 0.2 Initial Data Exploration

Let's examine the structure and basic properties of our dataset.

In [ ]:
# Check dataset dimensions
dim(dataset)

In [ ]:
# Examine data types of all variables
str(dataset)

### 0.3 Variable Classification

We separate our features into numerical and categorical variables for appropriate preprocessing.

In [ ]:
# Define set of numerical and categorical variables
num_var <- names(select_if(dataset %>% select(-y), is.numeric))
cat_var <- names(select_if(dataset %>% select(-y), function(x) is.character(x) | is.factor(x)))

In [ ]:
print("Numerical variables:")
print(num_var)

In [ ]:
print("Categorical variables:")
print(cat_var)

### 0.4 Data Quality Assessment

In [ ]:
# Check for missing values
sapply(dataset, function(x) any(is.na(x)))

In [ ]:
# Get basic statistics for numerical variables
summary(dataset[num_var])

### 0.5 Exploratory Data Analysis

Let's visualize the distribution and variability of our numerical features.

In [ ]:
# Check dispersion with box plot
box_plot <- function(df, standardize = TRUE) {
  "Create box plots for numerical variables to assess distribution and outliers"
  
  if (standardize) {
    # Standardize columns for better visualization
    df_scaled <- as.data.frame(scale(df))
  } else {
    df_scaled <- df
  }
  
  # Reshape data for ggplot
  df_long <- df_scaled %>%
    mutate(index = row_number()) %>%
    pivot_longer(-index, names_to = "variable", values_to = "value")
  
  # Create box plot
  p <- ggplot(df_long, aes(x = value, y = variable)) +
    geom_boxplot() +
    labs(title = ifelse(standardize, "Note that variables are standardized\nfor better visualization", "Box Plot of Variables"),
         x = "", y = "") +
    theme_minimal() +
    theme(plot.title = element_text(size = 16))
  
  return(p)
}

box_plot(dataset[num_var], standardize = TRUE)

### 0.6 Feature Selection

We remove certain variables that might cause data leakage or have too many missing values.

In [ ]:
# Remove variables that could cause data leakage or have quality issues
# duration: call duration is only known after the call ends (leakage)
# pdays: has many missing values (999 = not contacted)
# age, campaign, previous: removing for simplicity in this example
dataset <- dataset %>% select(-duration, -pdays, -age, -campaign, -previous)
num_var <- names(select_if(dataset %>% select(-y), is.numeric))
print("Remaining numerical variables:")
print(num_var)

### 0.7 Target Variable Analysis

Understanding our binary target variable is crucial for binary classification.

In [ ]:
# Check distribution for target variable (y: will the client subscribe?)
ggplot(dataset, aes(x = y)) +
  geom_bar(fill = "steelblue", color = "black") +
  labs(title = "Distribution of Target Variable",
       x = "Subscription (y)", y = "Count") +
  theme_minimal()

### 0.8 Addressing Class Imbalance

The dataset shows significant class imbalance (many more 'no' than 'yes' responses). We'll use undersampling to create a more balanced dataset for better model training.

In [ ]:
# Dataset is very unbalanced so we remove some observations for y=0 to be equal to 2*size of y=1.
# This is called "undersampling" - a technique to handle class imbalance

# We keep all y=1 (positive cases)
data_1 <- dataset %>% filter(y == 1)
cat("Positive cases (y=1):", nrow(data_1), "\n")

# We take y=0 as double the size of data_1 (2:1 ratio)
all_data_0 <- dataset %>% filter(y == 0)
percentage_corresponding_to_double_size <- (2 * nrow(data_1)) / nrow(all_data_0)

# Randomly sample negative cases
set.seed(0)
sample_indices <- sample(nrow(all_data_0), size = 2 * nrow(data_1))
data_0_small <- all_data_0[sample_indices, ]

cat("Negative cases - original:", nrow(all_data_0), ", sampled:", nrow(data_0_small), "\n")

In [ ]:
# Combine positive and sampled negative cases
dataset <- bind_rows(data_1, data_0_small)
cat("Final balanced dataset shape:", nrow(dataset), "x", ncol(dataset), "\n")

In [ ]:
# Verify the new class distribution after undersampling
ggplot(dataset, aes(x = y)) +
  geom_bar(fill = "steelblue", color = "black") +
  labs(title = "Target Variable Distribution After Undersampling",
       x = "Subscription (y)", y = "Count") +
  theme_minimal()

### 0.9 Feature Distribution Analysis by Target

Let's examine how our numerical features are distributed across the two target classes.

In [ ]:
# Plot the distribution of numerical variables by target class
# This helps us understand which features might be predictive

if(length(num_var) > 0) {
  # Reshape data for plotting
  plot_data <- dataset %>%
    select(all_of(num_var), y) %>%
    pivot_longer(-y, names_to = "variable", values_to = "value")
  
  # Create density plots
  p <- ggplot(plot_data, aes(x = value, fill = factor(y), alpha = 0.7)) +
    geom_density() +
    facet_wrap(~variable, scales = "free") +
    labs(title = "Distribution of Numerical Variables by Target Class",
         fill = "Subscribed") +
    scale_fill_discrete(labels = c("Not Subscribed (0)", "Subscribed (1)")) +
    theme_minimal() +
    theme(legend.position = "bottom")
  
  print(p)
} else {
  cat("No numerical variables to plot\n")
}

### 0.10 Categorical Variable Analysis

Now let's examine how categorical variables relate to our target variable.

In [ ]:
# Check the distribution of categorical variables by target class
# This shows the proportion of positive/negative outcomes within each category

if(length(cat_var) > 0) {
  # Create plots for each categorical variable
  plot_list <- list()
  
  for(var in cat_var) {
    # Calculate proportions
    plot_data <- dataset %>%
      group_by(!!sym(var), y) %>%
      summarise(count = n(), .groups = "drop") %>%
      group_by(!!sym(var)) %>%
      mutate(prop = count / sum(count))
    
    p <- ggplot(plot_data, aes(x = !!sym(var), y = prop, fill = factor(y))) +
      geom_bar(stat = "identity") +
      labs(title = paste("Target distribution for", var),
           x = var, y = "Proportion", fill = "Subscribed") +
      scale_fill_discrete(labels = c("Not Subscribed (0)", "Subscribed (1)")) +
      theme_minimal() +
      theme(axis.text.x = element_text(angle = 45, hjust = 1),
            legend.position = "bottom")
    
    plot_list[[length(plot_list) + 1]] <- p
  }
  
  # Display plots
  if(length(plot_list) > 0) {
    do.call(grid.arrange, c(plot_list, ncol = 2))
  }
} else {
  cat("No categorical variables to plot\n")
}

In [ ]:
head(dataset)

### 0.11 Data Preprocessing

We need to prepare our data for machine learning algorithms by encoding categorical variables and standardizing numerical features.

In [ ]:
# Create dummy variables for categorical features & standardize numerical features
# We'll use model.matrix to create dummy variables
dataset_dummy <- dataset

# Convert categorical variables to factors
for(var in cat_var) {
  dataset_dummy[[var]] <- as.factor(dataset_dummy[[var]])
}

# Create model matrix (automatically creates dummy variables)
if(length(c(cat_var, num_var)) > 0) {
  X_matrix <- model.matrix(y ~ ., data = dataset_dummy)[, -1]  # Remove intercept
  dataset_dummy <- as.data.frame(X_matrix)
  dataset_dummy$y <- dataset$y
}

# Standardize numerical variables (mean=0, std=1) for better algorithm performance
if(length(num_var) > 0) {
  for(var in num_var) {
    if(var %in% names(dataset_dummy)) {
      dataset_dummy[[var]] <- scale(dataset_dummy[[var]])[, 1]
    }
  }
}

In [ ]:
head(dataset_dummy)

### 0.12 Correlation Analysis

Let's examine correlations between variables to identify potential multicollinearity issues.

In [ ]:
# Calculate correlation matrix to identify highly correlated features
corrmat <- cor(dataset_dummy, use = "complete.obs")

In [ ]:
# Extract lower triangular correlations
corrmat_lower <- corrmat
corrmat_lower[upper.tri(corrmat_lower, diag = TRUE)] <- NA

# Convert to data frame and sort by absolute correlation
corrmat_df <- as.data.frame(as.table(corrmat_lower))
corrmat_df <- corrmat_df[!is.na(corrmat_df$Freq), ]
corrmat_df$abs_corr <- abs(corrmat_df$Freq)
corrmat_df <- corrmat_df[order(-corrmat_df$abs_corr), ]

# Show highest correlations
head(corrmat_df[, c("Var1", "Var2", "Freq")], 10)

In [ ]:
# Plot correlation heatmap
corrplot(corrmat, method = "color", type = "upper", order = "hclust", 
         tl.cex = 0.8, tl.col = "black", tl.srt = 45)

### 0.13 Feature Selection

We remove highly correlated features to avoid multicollinearity problems in our models.

In [ ]:
# Remove highly correlated features to avoid multicollinearity
dataset_original <- dataset  # save original dataset for reference

# Identify columns to drop based on high correlation (you may need to adjust these based on your data)
# This is a simplified approach - in practice, you'd want to examine the correlation matrix more carefully
high_corr_pairs <- corrmat_df[corrmat_df$abs_corr > 0.8 & corrmat_df$Var1 != corrmat_df$Var2, ]

if(nrow(high_corr_pairs) > 0) {
  cat("High correlation pairs found:\n")
  print(head(high_corr_pairs, 10))
  
  # For simplicity, we'll keep the dataset as is, but in practice you'd remove some variables
  # col_to_drop <- c("var1", "var2")  # Specify which variables to drop
  # dataset <- dataset_dummy %>% select(-all_of(col_to_drop))
} else {
  cat("No high correlation pairs found (correlation > 0.8)\n")
}

# Use the dummy dataset
dataset <- dataset_dummy

In [ ]:
# Ready to train and test our models!
X <- dataset %>% select(-y)
y <- dataset$y

cat("Feature matrix dimensions:", nrow(X), "x", ncol(X), "\n")
cat("Target vector length:", length(y), "\n")

In [ ]:
# Split train and test set
set.seed(0)
train_indices <- createDataPartition(dataset$y, p = 0.8, list = FALSE)
X_train <- X[train_indices, ]
X_test <- X[-train_indices, ]
y_train <- y[train_indices]
y_test <- y[-train_indices]

cat("Training set:", nrow(X_train), "x", ncol(X_train), "\n")
cat("Test set:", nrow(X_test), "x", ncol(X_test), "\n")
cat("Training labels:", length(y_train), "\n")
cat("Test labels:", length(y_test), "\n")

## 1. Linear Probability Model (LPM)

Before diving into logistic regression, let's start with the simpler Linear Probability Model. The LPM treats the binary dependent variable as if it were continuous and applies ordinary least squares (OLS) regression.

**Model specification:** P(y=1|X) = β₀ + β₁X₁ + β₂X₂ + ... + βₖXₖ + ε

While conceptually simple, the LPM has several important limitations that we'll explore.

In [ ]:
# Fit Linear Probability Model using OLS
# In R, we use lm() for linear regression

# Prepare data for lm()
train_data <- X_train
train_data$y <- y_train

# Train Linear Probability Model
lpm_model <- lm(y ~ ., data = train_data)

# Get predictions
y_train_pred_lpm <- predict(lpm_model, X_train)
y_test_pred_lpm <- predict(lpm_model, X_test)

# Calculate R-squared
train_r2 <- summary(lpm_model)$r.squared
test_r2 <- cor(y_test, y_test_pred_lpm)^2

cat(sprintf("LPM Training R²: %.4f\n", train_r2))
cat(sprintf("LPM Test R²: %.4f\n", test_r2))

In [ ]:
# Examine LPM predictions and identify problems
cat("Linear Probability Model - Prediction Statistics:\n")
cat(sprintf("Training set predictions - Min: %.4f, Max: %.4f\n", 
            min(y_train_pred_lpm), max(y_train_pred_lpm)))
cat(sprintf("Test set predictions - Min: %.4f, Max: %.4f\n", 
            min(y_test_pred_lpm), max(y_test_pred_lpm)))

cat("\nPredictions outside [0,1] range:\n")
train_outside <- sum(y_train_pred_lpm < 0 | y_train_pred_lpm > 1)
test_outside <- sum(y_test_pred_lpm < 0 | y_test_pred_lpm > 1)

cat(sprintf("Training: %d out of %d (%.1f%%)\n", 
            train_outside, length(y_train_pred_lmp), 
            100 * train_outside / length(y_train_pred_lpm)))
cat(sprintf("Test: %d out of %d (%.1f%%)\n", 
            test_outside, length(y_test_pred_lpm), 
            100 * test_outside / length(y_test_pred_lpm)))

In [ ]:
# Visualize LPM predictions vs actual values
library(gridExtra)

# Plot 1: Histogram of predicted probabilities
p1 <- ggplot(data.frame(pred = y_test_pred_lpm), aes(x = pred)) +
  geom_histogram(bins = 30, fill = "lightblue", color = "black", alpha = 0.7) +
  geom_vline(xintercept = 0, color = "red", linetype = "dashed", size = 1) +
  geom_vline(xintercept = 1, color = "red", linetype = "dashed", size = 1) +
  labs(title = "Distribution of LPM Predicted Probabilities",
       x = "Predicted Probability", y = "Frequency") +
  theme_minimal()

# Plot 2: Scatter plot of predictions vs actual
p2 <- ggplot(data.frame(pred = y_test_pred_lpm, actual = y_test), aes(x = pred, y = actual)) +
  geom_point(alpha = 0.6) +
  geom_abline(intercept = 0, slope = 1, color = "red", linetype = "dashed") +
  labs(title = "LPM: Predicted vs Actual Values",
       x = "Predicted Probability (LPM)", y = "Actual Value") +
  theme_minimal()

grid.arrange(p1, p2, ncol = 2)

In [ ]:
# Convert LPM predictions to binary classifications (using 0.5 threshold)
y_test_pred_lpm_binary <- ifelse(y_test_pred_lpm >= 0.5, 1, 0)

# Calculate accuracy
lpm_accuracy <- mean(y_test == y_test_pred_lpm_binary)
cat(sprintf("LPM Classification Accuracy: %.4f\n", lmp_accuracy))

## Problems with the Linear Probability Model

The Linear Probability Model has several fundamental issues when dealing with binary dependent variables:

### 1. **Predicted probabilities outside [0,1] range**
- As we saw above, LPM can predict negative probabilities or probabilities greater than 1
- This violates the basic definition of probability

### 2. **Heteroskedasticity**
- The error variance is not constant: Var(ε|X) = P(X)[1-P(X)]
- This violates the OLS assumption of homoskedasticity
- Standard errors are biased, affecting hypothesis testing

### 3. **Linear relationship assumption**
- LPM assumes a linear relationship between X and P(y=1|X)
- In reality, the effect of explanatory variables on probability is often non-linear
- Marginal effects are constant across all values of X (unrealistic)

### 4. **Distributional assumptions**
- OLS assumes normally distributed errors
- With binary outcomes, errors follow a Bernoulli distribution

### 5. **Efficiency concerns**
- Due to heteroskedasticity, OLS estimators are not efficient
- Maximum likelihood estimation (as in logistic regression) is more efficient

**Solution:** Use logistic regression, which addresses these issues by:
- Ensuring predicted probabilities stay within [0,1]
- Using the logistic function to model non-linear relationships
- Employing maximum likelihood estimation
- Properly handling the binary nature of the dependent variable

## 2. Logistic Regression

Now let's implement logistic regression, which addresses the limitations of the Linear Probability Model.

**Model specification:** 
- P(y=1|X) = 1 / (1 + e^(-(β₀ + β₁X₁ + β₂X₂ + ... + βₖXₖ)))
- This ensures probabilities remain between 0 and 1
- The relationship between X and P(y=1|X) is non-linear and S-shaped

In [ ]:
# Fit Logistic Regression using glm()
# In R, we use glm() with family="binomial" for logistic regression

# Train Logistic Regression Model
logit_model <- glm(y ~ ., data = train_data, family = binomial)

# Get summary
summary(logit_model)

In [ ]:
# Get fitted values on test set for logistic regression
y_test_predicted_prob_logit <- predict(logit_model, X_test, type = "response")
y_test_predicted_logit <- ifelse(y_test_predicted_prob_logit >= 0.5, 1, 0)

# Compare LPM vs Logistic Regression predictions
comparison_df <- data.frame(
  True = y_test,
  LPM_prob = y_test_pred_lpm,
  LPM_pred = y_test_pred_lpm_binary,
  Logit_prob = y_test_predicted_prob_logit,
  Logit_pred = y_test_predicted_logit
)

print(head(comparison_df, 20))

cat("\nModel Comparison:\n")
cat(sprintf("LPM Accuracy: %.4f\n", mean(y_test == y_test_pred_lpm_binary)))
cat(sprintf("Logistic Regression Accuracy: %.4f\n", mean(y_test == y_test_predicted_logit)))

In [ ]:
# Evaluate confusion matrix for Logistic Regression
library(caret)
confusionMatrix(factor(y_test_predicted_logit), factor(y_test))

In [ ]:
# Create and plot confusion matrix
plot_confusion_matrix <- function(y_true, y_pred, title = "Confusion Matrix") {
  # Create confusion matrix
  cm <- table(Predicted = y_pred, Actual = y_true)
  
  # Convert to data frame for ggplot
  cm_df <- as.data.frame(cm)
  
  # Create plot
  p <- ggplot(cm_df, aes(x = Actual, y = Predicted, fill = Freq)) +
    geom_tile(color = "white") +
    geom_text(aes(label = Freq), size = 12, color = "white") +
    scale_fill_gradient(low = "lightblue", high = "darkblue") +
    labs(title = title, x = "Actual", y = "Predicted") +
    theme_minimal() +
    theme(legend.position = "none")
  
  return(p)
}

# Plot confusion matrix for Logistic Regression
plot_confusion_matrix(y_test, y_test_predicted_logit, "Logistic Regression Confusion Matrix")

In [ ]:
# Evaluate precision, recall, F1-score
library(caret)

# Calculate metrics
cm <- confusionMatrix(factor(y_test_predicted_logit), factor(y_test), positive = "1")
print(cm)

In [ ]:
# Evaluate ROC curve
library(pROC)
library(ROCR)

# Calculate ROC curve
roc_obj <- roc(y_test, y_test_predicted_prob_logit)
auc_value <- auc(roc_obj)

# Plot ROC curve
plot(roc_obj, main = "Receiver Operating Characteristic", 
     col = "blue", lwd = 2)
abline(a = 0, b = 1, col = "red", lty = 2)
legend("bottomright", 
       legend = paste("Logistic Regression (AUC =", round(auc_value, 2), ")"),
       col = "blue", lwd = 2)

## 3. Exercises

Now it's time to practice! These exercises will help you understand the concepts better and gain hands-on experience with both Linear Probability Models and Logistic Regression.

### Exercise 1: Understanding Model Predictions

**Task:** Compare the predictions from both models and understand when they differ most.

**Instructions:**
1. Create a scatter plot comparing LPM probabilities vs Logistic Regression probabilities
2. Identify cases where the models disagree the most
3. Discuss what this tells us about the models

In [ ]:
# Exercise 1: Your code here
# Hint: Use ggplot() to plot LPM probabilities vs Logistic probabilities
# Add a diagonal line to show where predictions would be equal

# Your solution:


### Exercise 2: Threshold Analysis

**Task:** Explore how different classification thresholds affect model performance.

**Business Context:** In marketing, you might want to be more conservative (higher threshold) or more aggressive (lower threshold) in targeting customers.

**Instructions:**
1. Test thresholds of 0.3, 0.5, and 0.7 for the logistic regression model
2. Calculate accuracy, precision, and recall for each threshold
3. Discuss which threshold you would choose for a marketing campaign and why

In [ ]:
# Exercise 2: Your code here
# Hint: Use different thresholds to convert probabilities to binary predictions
# Calculate metrics using caret or custom functions

thresholds <- c(0.3, 0.5, 0.7)

# Your solution:


### Exercise 3: Feature Importance Analysis

**Task:** Understand which features are most important for predicting customer subscription.

**Business Context:** As a marketing manager, you want to know which customer characteristics are most predictive of subscription.

**Instructions:**
1. Extract and visualize the coefficients from the logistic regression model
2. Identify the top 5 most important features (largest absolute coefficients)
3. Interpret what these coefficients mean in business terms

In [ ]:
# Exercise 3: Your code here
# Hint: Use coef(logit_model) to get coefficients
# Create a bar plot showing feature importance

# Your solution:


### Exercise 4: New Customer Prediction

**Task:** Use your trained model to predict subscription probability for new customers.

**Business Context:** You have information about potential new customers and want to predict their likelihood of subscribing.

**Instructions:**
1. Create profiles for 3 hypothetical customers with different characteristics
2. Use both LPM and Logistic Regression to predict their subscription probabilities
3. Rank the customers by likelihood to subscribe

In [ ]:
# Exercise 4: Create new customer data
# Note: You'll need to ensure the new data has the same structure as your training data

# Example customer profiles (you'll need to adjust based on your features)
# Customer 1: Young, single, student
# Customer 2: Middle-aged, married, management job
# Customer 3: Retired, divorced

# Your solution:


### Exercise 5: Model Validation with Cross-Validation

**Task:** Assess how robust your model is using cross-validation.

**Business Context:** Before deploying a model in production, you want to ensure it performs consistently across different data samples.

**Instructions:**
1. Use 5-fold cross-validation to evaluate your logistic regression model
2. Calculate the mean and standard deviation of accuracy scores
3. Compare this with a simple baseline model that always predicts the majority class

In [ ]:
# Exercise 5: Your code here
# Hint: Use trainControl and train from caret package

# Your solution:


### Exercise 6: Business Impact Analysis

**Task:** Calculate the potential business impact of using your model.

**Business Context:** You want to quantify the value of using predictive modeling for customer targeting.

**Scenario:** 
- Cost of contacting a customer: €5
- Revenue from a successful subscription: €100
- You have 10,000 potential customers to contact

**Instructions:**
1. Calculate the profit from contacting all customers (no model)
2. Calculate the profit from using your model to select only high-probability customers (threshold = 0.6)
3. Compare the two strategies

In [ ]:
# Exercise 6: Business impact calculation

# Given parameters
contact_cost <- 5  # euros
subscription_revenue <- 100  # euros
total_customers <- 10000

# Use your test set to estimate performance
# Assume the test set is representative of the 10,000 customers

# Your solution:


### Exercise 7: Creating Synthetic Data (Advanced)

**Task:** Generate synthetic customer data and build a model from scratch.

**Business Context:** Understanding how to create realistic synthetic data helps in testing models and understanding data relationships.

**Instructions:**
1. Create a synthetic dataset with 1000 customers
2. Include features like: age, income, education_level, previous_purchases
3. Create a logical relationship where subscription probability depends on these features
4. Train both LPM and Logistic Regression on this data
5. Compare their performance

In [ ]:
# Exercise 7: Create synthetic data
# Hint: Use runif(), rnorm() and sample() functions to generate realistic customer data

set.seed(42)  # For reproducibility

# Your solution:
# 1. Generate customer features
# 2. Create a logical subscription probability based on features
# 3. Generate binary outcomes based on these probabilities
# 4. Train and compare models


### Reflection Questions

After completing the exercises, consider these questions:

1. **When would you prefer LPM over Logistic Regression?** 
   - Consider interpretability, computational complexity, and prediction quality

2. **How would you explain the difference between these models to a non-technical business stakeholder?**
   - Focus on practical implications rather than mathematical details

3. **What are the key business considerations when choosing a classification threshold?**
   - Think about costs of false positives vs false negatives

4. **How might you improve model performance further?**
   - Consider feature engineering, different algorithms, or ensemble methods

5. **What ethical considerations should you keep in mind when using these models for customer targeting?**
   - Think about fairness, privacy, and potential discrimination

### Additional Resources

For further learning:
- Practice with different datasets (e.g., credit approval, employee retention)
- Explore other classification algorithms (Random Forest, SVM)
- Learn about feature selection and engineering techniques
- Study advanced evaluation metrics (AUC-ROC, precision-recall curves)
- Investigate class imbalance handling techniques (SMOTE, cost-sensitive learning)